# Setting Up Cline with Ollama Cloud — Student Guide

*A hands-on guide for running an AI coding agent in your terminal.*

---

## What you're setting up (read this first)

**Cline** is an open-source AI coding *agent*. Unlike a chatbot, it can read your whole repository, create and edit files, run terminal commands, and show you a diff of every change before it happens. You drive it from the command line (the CLI).

**Ollama** is the engine behind Cline. We'll use **Ollama Cloud** — the model runs on Ollama's servers, so there's nothing big to download and even a modest laptop or a small Codespace can use a powerful model. You just need a free account on <https://ollama.com>.

> **The golden rule of agents:** Cline always asks for your confirmation before it changes a file or runs a command. Read what it proposes before approving. Nothing happens without your "yes." 

## How to use this guide

The commands live in **code cells** so you can copy them in one click — hover over a cell and use the **copy icon** in its top-right corner.

⚠️ **Run every command in a terminal**, not by executing the cell. Your own terminal on desktop, or the Codespace terminal. (The notebook kernel is Python, so running a command cell here would just throw a harmless error.)

Pick the setup that matches you:

- **Setup A — Your own PC** (Windows / macOS / Linux)
- **Setup B — GitHub Codespaces** (browser-based, matches our toolchain)

Both use the same Ollama Cloud models, so the steps are nearly identical.

---
# Choosing a model (and not burning your free limits)

Ollama Cloud's free tier doesn't charge by tokens — it charges by **GPU time**, which depends on how big the model is and how long the request runs. So the easiest way to make the free limits last is to **pick a light model** and only reach for a heavy one when you're truly stuck. Each model page shows a **usage level** from 1 (small/light) to 4 (extra heavy).

Free-tier limits reset on a **5-hour session window** and a **7-day weekly window**, and Free runs **one model at a time**.

Try these, lightest first. The `:cloud` tag is what you pass to `--model`:

| Model (`--model ...`) | Usage level | Good for | Notes |
|---|---|---|---|
| `gpt-oss:20b-cloud` | 1 — light ✅ | General coding & agent tasks | **Best default.** Cheapest on limits; plenty for bootcamp work. |
| `qwen3-coder:30b-cloud` | ~1–2 — light | Code-heavy tasks | Coding specialist; efficient MoE (~3B active). |
| `gemma4:31b-cloud` | 2 — medium | Coding + reading images/docs | Google's model, native tool-calling, strong coding scores. Burns a bit faster. |
| `gemini-3-flash-preview` | light–medium | Fast everyday help | Built for speed "at a fraction of the cost." |
| `kimi-k2.7-code` / `glm-5.2` / `minimax-m3` / `deepseek-v4-pro` | 3–4 — heavy ⚠️ | Hard, long problems only | Frontier quality, but drain a session fast. Use sparingly. |

**Tips to stretch your quota**

- Keep tasks small and specific — shorter requests use less GPU time.
- Stay in one session so cached context is reused.
- Check usage anytime at <https://ollama.com/settings> (email warning at 90%).
- Skip the parallel **Kanban** board (`cline kanban`) on Free — it wants several models at once, but Free allows only one.
- 💡 **Running a model on your own hardware is unlimited and free.** If your laptop can handle it, a small *local* Gemma (`ollama pull gemma4:e4b` or `:12b`) costs **zero** cloud usage — great for practice. (Cloud is still the easy path for weak laptops and Codespaces.)

> Model names and usage levels change often. Confirm the current `:cloud` tag and its usage level on the model's page (e.g. <https://ollama.com/library/gpt-oss>) before relying on it.

---
# Setup A — Your own PC (Desktop)

### Step 1 — Install Node.js 22 or newer

Cline runs on Node. Check your version — if it prints `v22.x` or higher you're set; otherwise install the latest LTS from <https://nodejs.org>.

In [ ]:
node --version

### Step 2 — Install Ollama

- **Windows / macOS:** download the installer from <https://ollama.com/download> and run it, then skip to the version check.
- **Linux:** run the install script.

In [ ]:
curl -fsSL https://ollama.com/install.sh | sh

Confirm it worked:

In [ ]:
ollama --version

### Step 3 — Install Cline

Cline is an npm package. Install it globally:

In [ ]:
npm install -g cline

Confirm it's available:

In [ ]:
cline --version

### Step 4 — Sign in to Ollama Cloud

Cloud models need a free account on <https://ollama.com>. Sign in from the terminal — this opens your browser; confirm there and you're connected.

In [ ]:
ollama signin

### Step 5 — Launch Cline with a cloud model

This sets the provider to Ollama and uses a cloud model. We default to the lightest one so your free limits last — swap in any model from the **Choosing a model** table above.

In [ ]:
ollama launch cline --model gpt-oss:20b-cloud

### Step 6 — Give Cline its first task

Once the session is open, type a request **inside Cline** (not the shell), for example:

> Analyze this repository and explain its architecture.

Inside the UI: **Tab** switches Plan/Act mode, **Esc** closes a dialog, and **`/quit`** exits back to the terminal. Cline proposes file changes and shows diffs — **approve or reject each step.**

That first `ollama launch …` sets your provider and default model; after that you can just use `cline` directly. Fire a one-shot task from the shell:

In [ ]:
cline "summarize this repository"

Or open the **Kanban** board to run several agents in parallel. Exit the Cline UI first with `/quit`, then run:

In [ ]:
cline kanban

---
# Setup B — GitHub Codespaces

A Codespace is a small Linux container in the cloud, so we use **Ollama Cloud** models — nothing heavy to download. All commands run in the Codespace terminal.

We authenticate with an **API key stored as a Codespaces secret**. The browser-based `ollama signin` doesn't fit a headless Codespace, but a secret is injected as an environment variable automatically, and the Ollama service picks it up with no sign-in prompt. You set the secret **once**; every Codespace you open afterward is ready to go.

## One-time setup (do this once per repo)

### Step 1 — Create an Ollama Cloud API key

Go to <https://ollama.com/settings/keys> → **Add API key** → copy it (you only see it once).

### Step 2 — Save it as a Codespaces secret

On GitHub: your repo → **Settings → Secrets and variables → Codespaces → New repository secret**.
- **Name:** `OLLAMA_API_KEY`  (exactly this)
- **Value:** paste your key → **Add secret**

You should see "Repository secret added" and `OLLAMA_API_KEY` in the list. That's the only place the key lives — never put it in a file or commit.

> Already have a Codespace open from before you added the secret? **Stop and restart it** (or create a new one) so the secret gets injected — secrets are loaded when the container starts.

## Each new Codespace (quick — secret is already there)

### Step 3 — Open the Codespace and confirm the secret

In your repo: **Code → Codespaces → Create codespace on main**. In the terminal, confirm the secret arrived on its own — this should print `OLLAMA_API_KEY is set`:

In [ ]:
echo ${OLLAMA_API_KEY:+OLLAMA_API_KEY is set}

> If it prints nothing: check the secret name is exactly `OLLAMA_API_KEY`, and that this Codespace was created **after** the secret was added (otherwise stop/restart it).

Node already ships in Codespaces, so this is just a check (expect `v22+`):

In [ ]:
node --version

### Step 4 — Install Ollama and Cline

A fresh container doesn't have these yet, so install them (the key needs no setup — it's already in the environment):

In [ ]:
curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
npm install -g cline

### Step 5 — Start the Ollama service in the background

It inherits `OLLAMA_API_KEY` from the environment automatically — no `export`, no `ollama signin`:

In [ ]:
nohup ollama serve > /tmp/ollama.log 2>&1 &

Give it a second, then confirm it's up:

In [ ]:
curl -s http://localhost:11434 && echo "  <- Ollama is running"

### Step 6 — Launch Cline with a cloud model

It should open straight to the prompt with `gpt-oss:20b-cloud` in the status bar — **no sign-in step**. That's the secret working.

In [ ]:
ollama launch cline --model gpt-oss:20b-cloud

Type a request **inside Cline**, e.g. *"Analyze this repository and explain its architecture."* Use **Tab** for Plan/Act, **Esc** to close a dialog, **`/quit`** to exit.

After that first launch sets your model, you can use `cline` directly — one-shot task:

In [ ]:
cline "write unit tests for the utils module"

Or the **Kanban** board (exit the Cline UI with `/quit` first):

In [ ]:
cline kanban

### No secret? One-session fallback

If you can't set a repo secret, you can paste the key by hand for the current session **before** starting `ollama serve` (the service only reads it from its own environment):

```bash
export OLLAMA_API_KEY="paste-your-key-here"
```

If `ollama serve` was already running, restart it so it picks up the key:

```bash
kill %1 2>/dev/null; nohup ollama serve > /tmp/ollama.log 2>&1 &
```

> **Never paste an API key into code, notebooks, or commits.** Use the Codespaces secret (preferred) or this per-session export. If a key is ever exposed, delete it at ollama.com/settings/keys and create a new one.

### A note on usage limits

Ollama Cloud's free tier uses **usage limits** (a session window that resets every few hours and a weekly cap) rather than per-token billing — so no surprise bills, but heavy use can pause you until the window resets. Check the **Usage** tab in your ollama.com account. Paid plans raise the limits if you need more.

### Optional — auto-install on Codespace creation

Tired of running the install commands in every new Codespace? Add a file at **`.devcontainer/devcontainer.json`** in your repo so each Codespace installs Ollama + Cline and starts the service automatically (the `OLLAMA_API_KEY` secret is still injected on its own):

```json
{
  "image": "mcr.microsoft.com/devcontainers/javascript-node:22",
  "postCreateCommand": "curl -fsSL https://ollama.com/install.sh | sh && npm install -g cline",
  "postStartCommand": "nohup ollama serve > /tmp/ollama.log 2>&1 &"
}
```

After that, a fresh Codespace only needs: `ollama launch cline --model gpt-oss:20b-cloud`.

---
# Everyday Cline commands (cheat sheet)

| Command | What it does |
|---|---|
| `ollama signin` | Sign in for cloud models |
| `ollama launch cline --model <name>:cloud` | First launch: set provider to Ollama + a cloud model, open the UI |
| `cline` | Start an interactive session (reuses your saved provider/model) |
| `cline "task"` | Run a one-shot task |
| `cline kanban` | Open the Kanban board for parallel agents (run after `/quit`) |
| `/quit` (inside the UI) | Exit the Cline terminal UI |
| `cline auth` | Choose provider (Ollama) and model |
| `cline config` | View/edit current settings |
| `cline doctor` | Diagnose configuration problems |
| `cline --help` | Full list of commands and flags |

---
# Troubleshooting

| Symptom | Fix |
|---|---|
| `node: command not found` or version below 22 | Install Node 22+ from nodejs.org; re-open the terminal. |
| `cline: command not found` | Re-run `npm install -g cline`. |
| "Cannot connect to Ollama" / connection refused | Ollama isn't running. Desktop: open the Ollama app. Codespaces: re-run the background `ollama serve` command. |
| "You need to be signed in to run Cloud models" | Codespaces: set `OLLAMA_API_KEY` (Step 4) **and restart** `ollama serve` so it picks up the key. Desktop: run `ollama signin`. |
| `ollama signin` keeps prompting / "Exiting via interrupt" in a Codespace | The browser flow doesn't fit a headless container — use the `OLLAMA_API_KEY` method instead (Setup B, Step 4). |
| Cloud requests blocked / paused | You hit a usage-limit window. Check the Usage tab on ollama.com or wait for the reset. |
| First response is a bit slow | Normal — the cloud model spins up on first use. |
| Network error on install in Codespaces | Codespaces normally allows outbound HTTPS; retry, or check your org's network policy. |

---
# Teaching / safe-use notes

- **Review before you approve.** Cline shows a diff for every change and asks before running commands — read it. This is the core habit to build.
- **Work on a branch.** Especially with auto-approval, run agents on a clean Git branch so you can review and roll back.
- **Keep secrets out of prompts and code.** Don't paste API keys, passwords, or private data into tasks or commit them.
- **Cloud models send requests to Ollama's servers.** Choose your model and tasks with any data-sensitivity rules in mind.
- **Start small.** Begin with "explain this repo" or "write one test" before attempting multi-file refactors.

---

*Sources: official Ollama docs (docs.ollama.com — Cline CLI integration and Cloud) and Cline CLI docs (docs.cline.bot). Model names and limits change frequently; always check the live model library at ollama.com/library.*